# Model Context Protocol with LangChain

This optional notebook shows how LangChain connects to MCP servers, loads their tools, resources, and prompts, and exposes those capabilities through an agent. The examples use a small local MCP server in `support/mcp_ops_server.py`, connected over streamable HTTP, so the protocol flow stays inspectable end to end.

## Learning Objectives

At the end of this notebook, you should be able to:

- Connect to an MCP server with `MultiServerMCPClient` over streamable HTTP.
- Load tools, resources, and prompts from the server and read what each provides.
- Use MCP-backed tools through a LangChain agent.
- Compare the default stateless client with an explicit session.

MCP standardizes how an application exposes tools, resources, and prompts to an LLM-aware client. In the LangChain integration, the MCP server remains separate from the agent runtime, while LangChain adapts the remote capabilities into objects the agent can use directly.


## Integration map

```mermaid
flowchart LR
    U["User request"] --> A["LangChain agent"]
    A --> C["MultiServerMCPClient"]
    C --> S["Local MCP server"]
    S --> T["Tools"]
    S --> R["Resources"]
    S --> P["Prompts"]
```

The server remains an independent process. LangChain discovers its capabilities and exposes them through familiar agent and workflow interfaces.


In [ ]:
import asyncio
import socket
import subprocess
import sys
import time
from collections.abc import Coroutine
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.prompts import load_mcp_prompt
from langchain_mcp_adapters.resources import load_mcp_resources
from langchain_mcp_adapters.sessions import StreamableHttpConnection
from langchain_mcp_adapters.tools import load_mcp_tools

## Environment and helper utilities

The MCP client APIs are asynchronous. The helper below runs async coroutines in a separate thread so the notebook can stay synchronous and compatible with standard notebook execution.


The LangChain MCP adapter exposes asynchronous methods because MCP clients often communicate with separate server processes. Standard notebooks already run an event loop, so calling `asyncio.run(...)` directly in a cell can conflict with that loop. The helper starts a short-lived background thread, runs the async MCP call there, and passes the result back through a `Queue`.

This keeps the notebook cells synchronous while still using the async MCP client correctly. Application code can use native async functions directly when it already runs in an async framework.


In [ ]:
load_dotenv(".env")
model = init_chat_model("groq:openai/gpt-oss-20b", temperature=0)


def run_async[T](coro: Coroutine[Any, Any, T]) -> T:
    """Run a coroutine on a short-lived worker thread and return its result."""

    def run_in_thread() -> T:
        return asyncio.run(coro)

    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(run_in_thread).result()

## Inspect the local server

The support file below uses the MCP Python SDK's `FastMCP` server to expose two tools, one resource, and one prompt over streamable HTTP.


In [ ]:
SERVER_PATH = Path("support/mcp_ops_server.py").resolve()
print(SERVER_PATH)
print(SERVER_PATH.read_text())

## Configure the MCP client

`MultiServerMCPClient` accepts one or more server definitions. This notebook uses a local HTTP MCP server so the capabilities are available without relying on an external hosted endpoint.


The `ops` entry names one MCP server. `transport="streamable_http"` means the client connects to an MCP endpoint over streamable HTTP. The server is still local, but it runs as a separate process and exposes its MCP endpoint at `/mcp`.


In [ ]:
MCP_HOST = "127.0.0.1"
MCP_PORT = 8765
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"


def wait_for_mcp_server(host: str, port: int, timeout_seconds: float = 10.0) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=1):
                return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"MCP server did not become ready at {host}:{port}")


mcp_server_process = subprocess.Popen(
    [sys.executable, str(SERVER_PATH), "--transport", "streamable-http"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_mcp_server(MCP_HOST, MCP_PORT)

ops_connection: StreamableHttpConnection = {
    "transport": "streamable_http",
    "url": MCP_URL,
}
client = MultiServerMCPClient({"ops": ops_connection})
client

## Load MCP tools

The LangChain adapter converts each MCP tool into a LangChain tool object. Those tools can then be passed directly into `create_agent(...)`.


In [ ]:
mcp_tools = run_async(client.get_tools())
[tool.name for tool in mcp_tools]

The adapter converted the server's MCP tools into LangChain tools. Their names, `add` and `planned_change_window`, match the functions defined in `support/mcp_ops_server.py`. From here they can be passed straight into `create_agent(...)` like any local tool.

In [ ]:
mcp_tools[1].args

## Load MCP resources

Resources expose read-only data over the protocol. The adapter returns Blob objects, which can be read as text or bytes depending on the content type.


In [ ]:
resources = run_async(client.get_resources("ops"))
{
    "uris": [str(resource.metadata["uri"]) for resource in resources],
    "text": resources[0].as_string(),
}

Resources expose read-only data over the protocol. The server offers one, `memo://service-overview`, and its text is the short operational memo defined on the server (primary region, escalation path, rollback target, and the maintenance lead time). Unlike a tool, a resource is fetched, not called with arguments.

## Load MCP prompts

MCP prompts are reusable message templates exposed by the server. LangChain converts them into message objects that can be used directly in a chat workflow.


In [ ]:
prompt_messages = run_async(
    client.get_prompt(
        "ops",
        "release_summary",
        arguments={"service": "billing-api", "risk": "high"},
    )
)
[(message.type, message.content) for message in prompt_messages]

In [ ]:
model.invoke(prompt_messages).content

## Use MCP tools through an agent

The agent below receives MCP-backed tools exactly like local LangChain tools. Because these MCP tools are asynchronous, the invocation is executed with `ainvoke(...)` through the notebook's async helper.


In [ ]:
mcp_agent = create_agent(
    model=model,
    tools=mcp_tools,
    system_prompt="Use MCP tools when they provide the requested operational data.",
)

In [ ]:
mcp_result = run_async(
    mcp_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Use the available tools to calculate 12 plus 8 "
                        "and tell me the billing-api change window."
                    ),
                }
            ]
        }
    )
)

mcp_result["messages"][-1].content

The agent used the MCP-backed tools exactly like local ones: `add` returned 20 for 12 plus 8, and `planned_change_window` returned the billing-api window of 22:00 UTC. The only difference from earlier notebooks is that these tools executed in a separate server process, not inside the notebook.

## Inspect the agent trace

MCP tool calls show up in the agent trace just like local tool calls. The difference is that the execution happened in a separate MCP server process rather than inside the agent's process.


In [ ]:
def summarize_messages(messages):
    rows = []
    for index, message in enumerate(messages):
        rows.append(
            {
                "index": index,
                "type": type(message).__name__,
                "content": getattr(message, "content", None),
                "tool_calls": getattr(message, "tool_calls", None),
            }
        )
    return rows


summarize_messages(mcp_result["messages"])

## Use an explicit client session

The LangChain MCP docs note that `MultiServerMCPClient` is stateless by default: each call creates a fresh MCP session. When tighter control over session lifetime is needed, the client can open an explicit session and load tools, resources, or prompts against that session directly.


In [ ]:
async def load_session_examples():
    async with client.session("ops") as session:
        session_tools = await load_mcp_tools(session)
        session_resources = await load_mcp_resources(session)
        session_prompt = await load_mcp_prompt(
            session,
            "release_summary",
            arguments={"service": "search-api", "risk": "medium"},
        )
        return session_tools, session_resources, session_prompt


session_tools, session_resources, session_prompt = run_async(load_session_examples())
{
    "tool_names": [tool.name for tool in session_tools],
    "resource_count": len(session_resources),
    "prompt_messages": [(message.type, message.content) for message in session_prompt],
}

By default `MultiServerMCPClient` is stateless: each call opens a fresh MCP session. When you need tighter control over the session lifetime, you can open one explicitly and load tools, resources, and prompts against it. The result mirrors the stateless calls: the same two tools, one resource, and a `release_summary` prompt (here for `search-api` at medium risk).

In [ ]:
model.invoke(session_prompt).content

## Stop the local MCP server

The server was started as a local background process for this notebook. Stopping it keeps the environment clean after the examples finish.


In [ ]:
mcp_server_process.terminate()
try:
    mcp_server_process.wait(timeout=5)
except subprocess.TimeoutExpired:
    mcp_server_process.kill()
    mcp_server_process.wait(timeout=5)

mcp_server_process.returncode

`terminate()` sends `SIGTERM`, so the process ends with return code `-15`. Shutting the background server down keeps the environment tidy once the examples finish.

## Summary

In this notebook you:

- Connected to a local MCP server with `MultiServerMCPClient` over streamable HTTP.
- Loaded the server's tools, resources, and prompts and read what each returned.
- Used MCP-backed tools through an agent (12 plus 8 = 20, and the billing-api window of 22:00 UTC).
- Compared the default stateless client with an explicit session.

MCP extends an agent's tool surface without embedding those capabilities in the agent process, which keeps the reasoning runtime and the external servers cleanly separate.

## References & Further Reading

- [**LangChain MCP**](https://docs.langchain.com/oss/python/langchain/mcp): Connecting agents to Model Context Protocol servers.
- [**MCP Python SDK**](https://py.sdk.modelcontextprotocol.io/): Building MCP servers and clients in Python.